# CardingForums — 01. Ingeniería de datos

En este notebook vamos a construir los datasets limpios que usaremos en los análisis posteriores.

**¿Qué es ingeniería de datos en este contexto?**  
Es el proceso de transformar datos crudos (el SQL dump tal como salió del foro) en datos listos para analizar. Esto incluye:
- Detectar y limpiar valores inválidos
- Normalizar formatos (fechas, texto)
- Construir tablas derivadas que el análisis va a necesitar

El principio fundamental: **garbage in, garbage out**. Si no limpiamos los datos antes, los análisis van a producir resultados incorrectos con total confianza.

---

## 1. Setup

Mismo setup que en el notebook 00. Importamos las librerías y cargamos los datos crudos.

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
import json
from pathlib import Path
from tqdm.notebook import tqdm

from src.utils import list_forums, merge_tables, load_forum, DATA_DIR, RESULTS_DIR

RESULTS_DIR.mkdir(exist_ok=True)
CATEGORY = "Carding Forums"
forum_zips = list_forums(CATEGORY)
print(f"Foros a procesar: {len(forum_zips)}")

## 2. Carga con auto-detección de formato

`load_forum()` detecta automáticamente el encoding de cada dump. Carder.pro, por ejemplo, usa cp1251 (encoding cirílico) en lugar de UTF-8. El parser lo maneja internamente.

**¿Por qué importa el encoding?**  
Si leemos un archivo cp1251 como si fuera UTF-8, los caracteres rusos aparecen como símbolos sin sentido (mojibake). El texto quedaría inutilizable para el análisis semántico.

Cargamos las 4 tablas que necesitamos para el análisis completo.

> `load_forum()` y `merge_tables()` (`src/utils.py`) — la primera auto-detecta el formato de cada dump, la segunda concatena la misma tabla de los 10 foros en un único DataFrame con columna `forum` de origen. Ver [`src/README.md`](../src/README.md).

In [ ]:
all_forums = []
for path in tqdm(forum_zips, desc="Cargando foros"):
    dfs = load_forum(path, tables=["user", "post", "thread", "pmtext"])
    all_forums.append(dfs)

users_raw   = merge_tables(all_forums, "user")
posts_raw   = merge_tables(all_forums, "post")
threads_raw = merge_tables(all_forums, "thread")
pms_raw     = merge_tables(all_forums, "pmtext")

print(f"Datos cargados (sin limpiar):")
print(f"  users:   {len(users_raw):,} filas")
print(f"  posts:   {len(posts_raw):,} filas")
print(f"  threads: {len(threads_raw):,} filas")
print(f"  pms:     {len(pms_raw):,} filas")

## 3. Limpieza de la tabla `users`

Aplicamos las limpiezas identificadas en el notebook de reconocimiento.

### 3.1 Timestamps epoch-0

En Unix, las fechas se almacenan como segundos transcurridos desde el 1 de enero de 1970 (epoch). El valor `0` significa literalmente esa fecha — no es un dato real, es un placeholder vacío que el sistema llenó con el valor por defecto.

**¿Por qué filtramos año < 2000?**  
Estos foros empezaron a operar alrededor de 2009. Un usuario registrado en 1970 o en 1995 es con certeza un dato corrupto.

In [ ]:
users = users_raw.copy()

# 1. Convertir joindate de timestamp Unix a datetime
# unit='s' le dice a pandas que los números son segundos desde epoch
users["joindate"] = pd.to_datetime(users["joindate"], errors="coerce", unit="s", utc=True)

# Cuántos registros tienen fecha inválida
before = len(users)
invalid_dates = users["joindate"].isna() | (users["joindate"] < pd.Timestamp("2000-01-01", tz="UTC"))
print(f"Usuarios con joindate inválido (nulo o < 2000): {invalid_dates.sum():,}")
print(f"  → Los mantenemos pero marcamos el campo como NaT para no usarlo en análisis temporales")

# En lugar de eliminar filas, reemplazamos las fechas inválidas con NaT (Not a Time)
users.loc[invalid_dates, "joindate"] = pd.NaT

print(f"\nUsuarios totales: {len(users):,} (sin eliminar ninguno)")
print(f"Usuarios con joindate válido: {users['joindate'].notna().sum():,}")

### 3.2 Normalización de usernames

Los usernames son el identificador más básico de un usuario. Para poder compararlos y buscar duplicados, necesitamos normalizarlos:

- **lowercase**: `"ADMIN"` y `"admin"` son la misma persona
- **strip**: eliminar espacios al inicio y al final — `" vendedor "` y `"vendedor"` son lo mismo

Creamos una columna nueva `username_normalized` para no perder el original.

In [ ]:
# Normalizamos username para comparaciones, pero mantenemos el original
users["username_normalized"] = (
    users["username"]
    .str.lower()
    .str.strip()
)

# Normalizar también los campos de contacto que usaremos para correlación
for col in ["email", "icq", "skype"]:
    if col in users.columns:
        users[col] = users[col].str.lower().str.strip().replace("", pd.NA)

# Convertir userid a numérico (viene como string del SQL)
users["userid"] = pd.to_numeric(users["userid"], errors="coerce")
users["posts"]  = pd.to_numeric(users.get("posts", pd.Series(0, index=users.index)), errors="coerce").fillna(0).astype(int)

print(f"Columnas después de normalización: {list(users.columns)}")
print(f"\nEjemplo de normalización:")
users[["username", "username_normalized", "email", "userid", "posts"]].head()

### 3.3 Eliminar duplicados en users

Un mismo usuario puede aparecer duplicado si el dump SQL tiene filas repetidas (error de exportación). Definimos un duplicado como dos filas con el mismo `userid` y el mismo `forum`.

In [ ]:
before = len(users)
users = users.drop_duplicates(subset=["userid", "forum"], keep="first")
removed = before - len(users)

print(f"Duplicados eliminados (mismo userid + forum): {removed:,}")
print(f"Usuarios únicos finales: {len(users):,}")

## 4. Limpieza de la tabla `posts`

### 4.1 Timestamps y posts con fechas inválidas

Aplicamos la misma lógica que con users: convertir a datetime, marcar inválidos como NaT, y filtrar posts con timestamps claramente erróneos para el análisis temporal (pero mantenerlos para análisis de texto).

In [ ]:
posts = posts_raw.copy()

# Convertir dateline (timestamp Unix) a datetime UTC
posts["dateline"] = pd.to_datetime(posts["dateline"], errors="coerce", unit="s", utc=True)

# Marcar timestamps inválidos (< 2000 o > 2025)
valid_range = (
    posts["dateline"].between(
        pd.Timestamp("2000-01-01", tz="UTC"),
        pd.Timestamp("2025-12-31", tz="UTC")
    )
)
posts["dateline_valid"] = valid_range

print(f"Posts con dateline fuera de rango: {(~valid_range & posts['dateline'].notna()).sum():,}")
print(f"Posts con dateline nulo: {posts['dateline'].isna().sum():,}")
print(f"Posts con dateline válido: {valid_range.sum():,}")

### 4.2 Posts vacíos

Un post vacío es un post borrado (el contenido fue eliminado por el moderador, pero la fila en la base de datos quedó). No aportan nada al análisis semántico, y distorsionan los recuentos de actividad.

Creamos una columna `text_valid` para marcar qué posts tienen texto real.

In [ ]:
if "pagetext" in posts.columns:
    # Un post es válido si tiene texto de al menos 5 caracteres (sin espacios)
    posts["text_clean"] = posts["pagetext"].fillna("").str.strip()
    posts["text_valid"] = posts["text_clean"].str.len() >= 5

    empty_count = (~posts["text_valid"]).sum()
    valid_count = posts["text_valid"].sum()
    print(f"Posts con texto válido (≥5 chars): {valid_count:,}")
    print(f"Posts vacíos o muy cortos (descartados del análisis semántico): {empty_count:,}")
else:
    posts["text_valid"] = True
    print("Columna 'pagetext' no encontrada en este dump")

### 4.3 Normalizar userid en posts

El `userid` en la tabla posts tiene que ser del mismo tipo que en la tabla users para que los joins funcionen.

In [ ]:
posts["userid"]   = pd.to_numeric(posts["userid"], errors="coerce")
posts["threadid"] = pd.to_numeric(posts["threadid"], errors="coerce")
posts["postid"]   = pd.to_numeric(posts["postid"], errors="coerce")

# Eliminar duplicados (mismo postid + forum)
before = len(posts)
posts = posts.drop_duplicates(subset=["postid", "forum"], keep="first")
print(f"Duplicados eliminados de posts: {before - len(posts):,}")
print(f"Posts únicos finales: {len(posts):,}")

## 5. Resumen de la limpieza

Buena práctica: siempre documentar cuántos registros se modificaron en cada paso de limpieza. Esto permite auditar el proceso y detectar si algo fue demasiado agresivo.

In [ ]:
cleaning_report = {
    "users_raw": len(users_raw),
    "users_clean": len(users),
    "users_invalid_joindate": int(users["joindate"].isna().sum()),
    "posts_raw": len(posts_raw),
    "posts_clean": len(posts),
    "posts_invalid_date": int((~posts["dateline_valid"]).sum()),
    "posts_empty_text": int((~posts.get("text_valid", pd.Series(True, index=posts.index))).sum()),
}

print("REPORTE DE LIMPIEZA")
print("=" * 40)
for k, v in cleaning_report.items():
    print(f"  {k:35s}: {v:,}")

(RESULTS_DIR / "01_cleaning_report.json").write_text(
    json.dumps(cleaning_report, indent=2)
)

## 6. Preparación para análisis de red: tabla usuario↔thread

El análisis de red (notebook 02) necesita saber qué usuarios participaron en qué threads. Dos usuarios están "conectados" si participaron en el mismo thread — la hipótesis es que interactuaban (aunque sea indirectamente).

Construimos una tabla de **co-participación** que relaciona cada par `(userid, threadid)` con el número de posts que ese usuario hizo en ese thread.

**¿Por qué esto y no leer posts directamente en el análisis de red?**  
Porque los grafos se construyen a nivel de nodos y aristas. Necesitamos una tabla limpia de participación para construir la matriz de adyacencia eficientemente.

In [ ]:
# Solo usamos posts válidos (con texto real y fecha válida)
posts_for_network = posts[
    posts["userid"].notna() &
    posts["threadid"].notna() &
    posts.get("text_valid", pd.Series(True, index=posts.index))
].copy()

# Tabla de co-participación: cuántos posts hizo cada usuario en cada thread
user_thread = (
    posts_for_network
    .groupby(["forum", "userid", "threadid"])
    .size()
    .reset_index(name="post_count")
)

print(f"Tabla usuario↔thread:")
print(f"  Filas (combinaciones únicas usuario-thread): {len(user_thread):,}")
print(f"  Usuarios distintos: {user_thread['userid'].nunique():,}")
print(f"  Threads distintos: {user_thread['threadid'].nunique():,}")
print()
user_thread.head()

In [ ]:
# Guardar para uso en notebook 02
user_thread.to_parquet(RESULTS_DIR / "01_user_thread.parquet", index=False)
print("✓ Guardado: results/01_user_thread.parquet")

## 7. Preparación para análisis semántico: corpus por usuario

El análisis semántico (notebook 03) necesita el texto de cada usuario concatenado en un solo documento. Esto representa la "voz" del usuario — todo lo que escribió en el foro.

**¿Por qué concatenar todos los posts en un solo texto?**  
Los embeddings (vectores semánticos) que generaremos son más estables y representativos cuando tienen más texto. Un post aislado puede no capturar bien el estilo y los temas del usuario. El corpus completo sí.

Solo incluimos usuarios con al menos 10 posts, para que el corpus tenga suficiente señal.

In [ ]:
MIN_POSTS_FOR_EMBEDDING = 10

# Solo posts con texto válido
posts_with_text = posts[
    posts.get("text_valid", pd.Series(True, index=posts.index)) &
    posts["userid"].notna()
].copy()

# Contar posts por usuario (en cada foro)
post_count_per_user = posts_with_text.groupby(["forum", "userid"]).size().reset_index(name="n_posts")

# Usuarios con suficientes posts
eligible = post_count_per_user[post_count_per_user["n_posts"] >= MIN_POSTS_FOR_EMBEDDING]
print(f"Usuarios elegibles para embedding (≥{MIN_POSTS_FOR_EMBEDDING} posts): {len(eligible):,}")

# Construir corpus: concatenar todos los posts de cada usuario
corpus = (
    posts_with_text
    .merge(eligible[["forum", "userid"]], on=["forum", "userid"])
    .groupby(["forum", "userid"])["text_clean"]
    .apply(lambda texts: " ".join(texts))
    .reset_index(name="corpus")
)

# Añadir metadata de usuario
corpus = corpus.merge(
    users[["forum", "userid", "username", "posts"]],
    on=["forum", "userid"],
    how="left"
)

print(f"Corpus generados: {len(corpus):,}")
print(f"\nLongitud media del corpus (caracteres): {corpus['corpus'].str.len().mean():,.0f}")
corpus[["forum", "username", "posts", "corpus"]].head(3)

In [ ]:
# Guardar corpus para notebook 03
corpus.to_parquet(RESULTS_DIR / "01_user_corpus.parquet", index=False)
print("✓ Guardado: results/01_user_corpus.parquet")

## 8. Guardar los datos limpios

Guardamos las tablas `users` y `posts` limpias en formato Parquet. 

**¿Por qué Parquet y no CSV?**  
- Parquet preserva los tipos de datos (datetime, int, float). CSV los convierte todo a texto.
- Parquet es ~5-10x más rápido de leer y ocupa menos espacio.
- Para datasets de cientos de miles de filas, la diferencia es muy notable.

In [ ]:
users.to_parquet(RESULTS_DIR / "01_users_clean.parquet", index=False)
posts.to_parquet(RESULTS_DIR / "01_posts_clean.parquet", index=False)

print("Archivos guardados:")
print(f"  results/01_users_clean.parquet  — {len(users):,} usuarios")
print(f"  results/01_posts_clean.parquet  — {len(posts):,} posts")
print(f"  results/01_user_thread.parquet  — {len(user_thread):,} participaciones usuario-thread")
print(f"  results/01_user_corpus.parquet  — {len(corpus):,} corpus de usuario")
print()
print("→ Siguiente paso: 02_analisis_estructural.ipynb")